In [1]:
import sqlite3
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    Doc
)

from pymorphy2 import MorphAnalyzer
show_database_structure('articles.db')


# 1. Загрузка данных из SQLite базы данных
def load_data_from_db(db_file, table_name):
    conn = sqlite3.connect(db_file)
    df = pd.read_sql_query(f'SELECT description FROM {table_name}', conn)

    sentences = []
    for row in df['description']:
        sentences.extend(re.split(r'[.!?]+\s+', str(row)))

    sentences = [sent.strip() for sent in sentences if len(sent.strip()) > 0]
    return sentences

sentences = load_data_from_db('articles.db', 'articles')
print(f"Всего предложений: {len(sentences)}")


# 2. Синтаксический разбор и выделение подлежащих и сказуемых
def extract_subject_and_predicate(sentence, segmenter, morph_tagger, syntax_parser):
    doc = Doc(sentence)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.parse_syntax(syntax_parser)

    subject = None
    predicate = None

    for token in doc.sents[0].tokens:
        if token.rel == 'nsubj':
            subject = token.text.lower()
        elif token.rel == 'root':
            predicate = token.text.lower()

    return subject, predicate


# 3. Инициализация инструментов Natasha
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)

# 4. Получение и фильтрация пар подлежащее-сказуемое
pairs = [(extract_subject_and_predicate(sent, segmenter, morph_tagger, syntax_parser)) for sent in sentences]
filtered_pairs = [(sub, pred) for sub, pred in pairs if sub and pred]

print("ПРИМЕРЫ ПАР ПОДЛЕЖАЩЕЕ-СКАЗУЕМОЕ:")
for i, (subject, predicate) in enumerate(filtered_pairs[:10], 1):
    print(f"{i:2d}. {subject:25s} - {predicate}")


# 5. Подсчет частот встречаемости
co_occurrences = Counter(filtered_pairs)


# 6. Визуализация TOP-N сочетаний
def plot_top_pairs(co_occurrences, top_n=20):
    top_pairs = co_occurrences.most_common(top_n)
    subjects_predicates = [f"{sub} — {pred}" for sub, pred in top_pairs]
    frequencies = [freq for _, freq in top_pairs]

    plt.figure(figsize=(10, 6))
    plt.barh(subjects_predicates, frequencies, color='skyblue')
    plt.title(f"ТОП-{top_n} наиболее частых сочетаний подлежащих и сказуемых")
    plt.xlabel("Частота")
    plt.ylabel("Сочетания подлежащих и сказуемых")
    plt.gca().invert_yaxis()
    plt.show()


# Визуализируем результаты
plot_top_pairs(co_occurrences, top_n=20)


ModuleNotFoundError: No module named 'natasha'